# Day 06 — Data Preprocessing & Machine Learning Pipeline

This notebook prepares the engineered Titanic features for machine learning. It creates a reproducible train/test split, handles missing numerical values, scales the input features, and keeps preprocessing isolated from the test set to prevent data leakage.

The central engineering idea is simple: **learn preprocessing rules from training data only, then reuse those learned rules on unseen data.**

## 1. Load the Dataset & Prepare Model Features

The Day 05 work introduced binary encoding for `sex` and one-hot encoding for `embarked`. Here, those transformations are recreated explicitly so this notebook can run independently from a previous notebook session.

I keep only the columns required for this modeling exercise. Rows without a known target (`survived`) or required categorical values are excluded. Missing values in `age` and `fare` are deliberately retained because the Scikit-learn imputer will handle them inside the pipeline.

In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Load the original Titanic dataset
df = sns.load_dataset("titanic")

# Encode the binary sex feature
df["sex_encoded"] = df["sex"].map({"male": 1, "female": 0})

# One-hot encode embarked, using the first category as the reference
embarked_encoded = pd.get_dummies(
    df["embarked"],
    prefix="embarked",
    drop_first=True,
    dtype=int
)

# Guarantee a consistent feature schema
embarked_encoded = embarked_encoded.reindex(
    columns=["embarked_Q", "embarked_S"],
    fill_value=0
)

df = pd.concat([df, embarked_encoded], axis=1)

# Keep records with a known target and valid encoded categories
df = df.dropna(subset=["survived", "sex_encoded"])

features = [
    "pclass",
    "age",
    "fare",
    "sex_encoded",
    "embarked_Q",
    "embarked_S"
]

X = df[features].copy()
y = df["survived"].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
print("\nMissing values before preprocessing:")
print(X.isna().sum())

## 2. Split the Data into Training & Test Sets

The model must be evaluated on records it did not learn from. I use an **80/20 split**:

- Training set (80%): used to learn preprocessing parameters and fit models.
- Test set (20%): held back until evaluation.

`stratify=y` preserves approximately the same survival-class proportions in both subsets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nSurvival distribution in training data:")
print(y_train.value_counts(normalize=True).sort_index())
print("\nSurvival distribution in test data:")
print(y_test.value_counts(normalize=True).sort_index())

## 3. Build the Preprocessing Pipeline

A Scikit-learn `Pipeline` chains multiple transformations into one reusable workflow:

1. **SimpleImputer(strategy="median")** fills missing numerical values using the median learned from the training data.
2. **StandardScaler()** standardizes each feature using the training mean and standard deviation.

The key distinction is between `fit_transform` and `transform`:

- `fit_transform(X_train)` learns the imputation and scaling parameters from training data, then applies them.
- `transform(X_test)` applies those already-learned parameters without recalculating them.

If we fitted the imputer or scaler on the full dataset before splitting, information from the test set would influence training. That is a form of **data leakage** and can make evaluation results overly optimistic.

In [ ]:
preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Learn preprocessing parameters from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same learned transformations to unseen test data
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("\nRemaining missing values in processed training data:",
      pd.isna(X_train_processed).sum())
print("Remaining missing values in processed test data:",
      pd.isna(X_test_processed).sum())

## Engineering Takeaways

- Split the data before learning imputation or scaling parameters.
- Use stratification for classification tasks when preserving target proportions matters.
- Fit preprocessing steps on training data only; transform test data with the fitted pipeline.
- Median imputation is a robust baseline for numerical features, but it should still be validated against the data and problem.
- Standardization is especially useful for scale-sensitive algorithms such as Logistic Regression. Tree-based models generally do not require feature scaling.
- A pipeline makes preprocessing reproducible and reduces the chance of accidentally applying different transformations at training and inference time.

> **Production perspective:** The next improvement is to combine preprocessing and the estimator into one complete Scikit-learn pipeline. That makes the entire prediction workflow easier to persist, validate, and deploy.